In [ ]:
import json

# Label map
with open("int_mapping.json", 'r') as file_object:
    # Use json.load() to convert the file content into a Python dictionary
    label_map_dict = json.load(file_object)
inverted_map = {value: key for key, value in label_map_dict.items()} #use pre-training
num_classes = len(label_map_dict)

Logistic Regressor

In [ ]:
from PIL import Image
import cv2
import numpy as np
import os

# Extract features
def build_feature_vector(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # --- engineered features ---
    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.mean(edges) / 255.0

    avg_intensity = np.mean(gray) / 255.0

    y_coords, x_coords = np.indices(gray.shape)
    total = np.sum(gray) + 1e-8
    cx = np.sum(x_coords * gray) / total / gray.shape[1]
    cy = np.sum(y_coords * gray) / total / gray.shape[0]

    extra_features = np.array([edge_density, avg_intensity, cx, cy])

    # --- image features ---
    small = cv2.resize(gray, (16, 16))
    pixels = small.flatten() / 255.0   # normalize

    # --- combine ---
    return np.concatenate([pixels, extra_features])

# Images to arrays

X = []
y = []

root_dir = "extracted_images"

for label_name in os.listdir(root_dir):
    class_dir = os.path.join(root_dir, label_name)

    if not os.path.isdir(class_dir):
        continue

    for file in os.listdir(class_dir):
        img_path = os.path.join(class_dir, file)

        try:
            img = np.array(Image.open(img_path).convert("RGB"))
            features = build_feature_vector(img)
            X.append(features)
            mapped_label = inverted_map[label_name]
            print(label_name, ": ", mapped_label)
            y.append(int(mapped_label))  # folder name = label

        except Exception as e:
            print(f"Skipping {img_path}: {e}")

# Convert to NumPy arrays
X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)
# extract extra features

# train
regressor = cv2.ml.LogisticRegression_create()

Streaming output truncated to the last 5000 lines.
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ :  2
+ : 

In [ ]:
print("Total loaded:", X.shape)

Total loaded: (239641, 260)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import torch
# HPO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)
#regressor.fit(X_train, y_train)
scaler = StandardScaler()

# fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

X_train2, X_HPO, y_train2, y_HPO = train_test_split(X_train_scaled, y_train, test_size=.2, random_state=39)
print(X_HPO.shape)
param_grid = [
    {
        'penalty': ['l2'],
        'C': [0.1, 1, 10],
        'solver': ['lbfgs'],
        'max_iter': [200, 500]
    },
    {
        'penalty': ['l1'],
        'C': [0.1, 1],
        'solver': ['saga'],
        'max_iter': [200, 500]
    }
]
log_model = LogisticRegression()
clf = GridSearchCV(log_model,param_grid = param_grid, cv = 2, verbose=True,n_jobs=-1)
best_clf = clf.fit(X_HPO,y_HPO)
best_clf.best_estimator_

(38343, 260)
Fitting 2 folds for each of 10 candidates, totalling 20 fits


KeyboardInterrupt: 

In [ ]:
#Training
final_model = LogisticRegression(
    C=.1,          # adjust key name if needed
    penalty='l2',
    solver='lbfgs',
    max_iter=200
)

final_model.fit(X_train_scaled, y_train)

LogisticRegression(C=0.1, max_iter=200)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
import pickle
# validation
X_test_scaled = scaler.transform(X_test)

y_pred = final_model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print("Validation accuracy:", acc)

print(classification_report(y_test, y_pred))

# Save the model
with open('lr_model.pkl', 'wb') as file:
    pickle.dump(final_model, file)

Validation accuracy: 0.9079889002482839
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      2819
           1       0.96      0.98      0.97      2865
           2       0.90      0.92      0.91      5144
           3       0.99      1.00      1.00      6655
           4       0.95      0.96      0.96      1390
           5       0.87      0.91      0.89      5246
           6       0.93      0.92      0.92      5138
           7       0.92      0.93      0.93      2168
           8       0.83      0.76      0.79      1537
           9       0.89      0.82      0.85       719
          10       0.93      0.96      0.95       616
          11       0.89      0.82      0.85       568
          12       0.90      0.89      0.89       637
          13       0.90      0.90      0.90       755
          14       0.92      0.94      0.93      2634
          15       0.85      0.88      0.86      5411
          16       0.95      0.74      0.

CNN

In [ ]:
!unzip extracted_images.zip

Streaming output truncated to the last 5000 lines.
  inflating: extracted_images/y/exp69885.jpg  
  inflating: extracted_images/y/exp69903.jpg  
  inflating: extracted_images/y/exp69910.jpg  
  inflating: extracted_images/y/exp69911.jpg  
  inflating: extracted_images/y/exp69925.jpg  
  inflating: extracted_images/y/exp69927.jpg  
  inflating: extracted_images/y/exp69937.jpg  
  inflating: extracted_images/y/exp69948.jpg  
  inflating: extracted_images/y/exp69951.jpg  
  inflating: extracted_images/y/exp69997.jpg  
  inflating: extracted_images/y/exp70.jpg  
  inflating: extracted_images/y/exp70003.jpg  
  inflating: extracted_images/y/exp70103.jpg  
  inflating: extracted_images/y/exp70158.jpg  
  inflating: extracted_images/y/exp70164.jpg  
  inflating: extracted_images/y/exp70171.jpg  
  inflating: extracted_images/y/exp70176.jpg  
  inflating: extracted_images/y/exp70179.jpg  
  inflating: extracted_images/y/exp70200.jpg  
  inflating: extracted_images/y/exp70203.jpg  
  inflating:

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())

class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  # RGB input
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),  # correct for 64x64 input
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


#cnn_model = CNN(num_classes=22).to(device)

True


In [ ]:
# CNN HPO
import random
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, DataLoader
from torchvision import transforms
import torch.optim as optim
import os
import itertools

train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

criterion = nn.CrossEntropyLoss()
dataset = ImageFolder("extracted_images", transform=train_transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])


def train_model(model, loader, optimizer, epochs=3):
    criterion = nn.CrossEntropyLoss()
    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()*images.size(0)
            avg_loss = total_loss/len(loader.dataset)

        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")


    return avg_loss


# ---------------- HPO loop ----------------
best_loss = float("inf")
best_config = None
combos = []
i = 0

for lr, batch in itertools.product([.0001, .001, .01], [16, 32]):

    print(f"\nTrial {i}: lr={lr}, batch={batch}")

    train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=True)

    model = CNN(num_classes=len(dataset.classes)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    loss = train_model(model, train_loader, optimizer)

    print("Final loss:", loss)

    if loss < best_loss:
        best_loss = loss
        best_config = (lr, batch)
    i+=1

print("\nBest config:", best_config)

'\nfor lr, batch in itertools.product([.0001, .001, .01], [16, 32]):\n\n    print(f"\nTrial {i}: lr={lr}, batch={batch}")\n\n    train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=True)\n\n    model = CNN(num_classes=len(dataset.classes)).to(device)\n    optimizer = optim.Adam(model.parameters(), lr=lr)\n\n    loss = train_model(model, train_loader, optimizer)\n\n    print("Final loss:", loss)\n\n    if loss < best_loss:\n        best_loss = loss\n        best_config = (lr, batch)\n    i+=1\n\nprint("\nBest config:", best_config)\n'

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def train_one_epoch(epoch: int):
    print("Training...")
    model.train()
    running_loss = 0.0
    batch_num = 1

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        print("Batch #", batch_num)
        batch_num +=1
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        print("Batch loss: ", loss.item())

    avg_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch:02d} | Train loss: {avg_loss:.4f}")

data = datasets.ImageFolder("extracted_images", transform=train_transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_data, val_data = random_split(
    data,
    [train_size, val_size]
)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader  = DataLoader(val_data, batch_size=32)

model = CNN(num_classes=len(dataset.classes)).to(device)
optimizer = optim.Adam(model.parameters(), lr=.001)

def evaluate(misclassified_images):
    print("Evaluating...")
    model.eval()
    correct = 0
    total = 0

    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        preds = torch.argmax(logits, dim=1)
        incorrect_mask = preds != labels
        preds_cpu = preds.cpu()
        labels_cpu = labels.cpu()

        for i in range(len(images)):
            if incorrect_mask[i]:
                misclassified_images.append({
                    #'image': paths[i],
                    'true_label': labels_cpu[i].item(),
                    'pred_label': preds_cpu[i].item()
                })
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    print(f"Test accuracy: {acc*100:.2f}%")
    return acc


epoch_acc_arr = []
misclassified = []

In [ ]:
# training
for i in range(0, 10):
  train_one_epoch(i)
  epoch_acc = evaluate(misclassified)
  epoch_acc_arr.append(epoch_acc)


Streaming output truncated to the last 5000 lines.
Batch loss:  0.005392218008637428
Batch # 3494
Batch loss:  0.02207132615149021
Batch # 3495
Batch loss:  0.007554239127784967
Batch # 3496
Batch loss:  0.045341964811086655
Batch # 3497
Batch loss:  0.16095203161239624
Batch # 3498
Batch loss:  0.0008089840412139893
Batch # 3499
Batch loss:  0.0010020493064075708
Batch # 3500
Batch loss:  0.001985497772693634
Batch # 3501
Batch loss:  0.005378758534789085
Batch # 3502
Batch loss:  0.1405351608991623
Batch # 3503
Batch loss:  0.037656404078006744
Batch # 3504
Batch loss:  0.015750372782349586
Batch # 3505
Batch loss:  0.019206106662750244
Batch # 3506
Batch loss:  0.0003508994705043733
Batch # 3507
Batch loss:  0.001466603484004736
Batch # 3508
Batch loss:  0.0017963144928216934
Batch # 3509
Batch loss:  0.0010169415036216378
Batch # 3510
Batch loss:  0.02389436960220337
Batch # 3511
Batch loss:  0.18044191598892212
Batch # 3512
Batch loss:  0.002185824094340205
Batch # 3513
Batch loss

In [ ]:
print(data.class_to_idx)

{'(': 0, ')': 1, '+': 2, '-': 3, '0': 4, '1': 5, '2': 6, '3': 7, '4': 8, '5': 9, '6': 10, '7': 11, '8': 12, '9': 13, '=': 14, 'X': 15, 'div': 16, 'f': 17, 'forward_slash': 18, 'neq': 19, 'times': 20, 'y': 21}


In [ ]:
torch.save(model.state_dict(), 'cnn_model_weights.pth')
print(epoch_acc_arr)
print(misclassified)
print(min(epoch_acc_arr), max(epoch_acc_arr), epoch_acc_arr.mean())

NameError: name 'torch' is not defined